# DSPy Optimization — BootstrapFewShot vs MIPROv2

**Week 6 | Notebook 2 of 6**

**What you'll learn:**
- What is compilation? (conceptual walkthrough)
- Preparing trainset + devset (dspy.Example format)
- Writing a custom metric function
- Running BootstrapFewShot — inspect generated few-shot demos
- Running MIPROv2 — inspect generated instructions
- Comparing: baseline vs optimized on devset score
- Saving and reloading the optimized program

**Runtime:** ~45 minutes (API calls during optimization)

**Cost-saving:** Default 10 trials (reduce in .env with DSPY_OPTIMIZER_TRIALS)

In [ ]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/02_optimizers.ipynb")

## 1. Setup

In [ ]:
import dspy
from dspy.evaluate import Evaluate
from dspy.teleprompt import BootstrapFewShot, MIPROv2

from src.config import DSPY_OPTIMIZER_TRIALS, get_dspy_lm
from src.datasets import generate_qa_pairs

lm = get_dspy_lm()
dspy.configure(lm=lm)

print("✅ DSPy configured")
print(f"Optimizer trials: {DSPY_OPTIMIZER_TRIALS}")

## 2. What Is Compilation?

In [ ]:
# DSPy compilation = automatically improving your program
# by generating better instructions and few-shot examples

print("DSPy Compilation Flow:")
print("  1. Define your program (signatures + modules)")
print("  2. Provide training examples")
print("  3. Define a metric (0-1 scoring function)")
print("  4. Run optimizer (BootstrapFewShot / MIPROv2)")
print("  5. Optimized program has better prompts + demos")
print("  6. Evaluate on devset")
print("\n💡 Think of it like a compiler: Python → optimized prompts")

## 3. Preparing Trainset + Devset

In [ ]:
# Convert synthetic data to dspy.Example format
qa_data = generate_qa_pairs(40)

examples = [
    dspy.Example(question=d["question"], answer=d["expected"]).with_inputs("question")
    for d in qa_data
]

# Split: 70% train, 30% dev
split = int(len(examples) * 0.7)
trainset = examples[:split]
devset = examples[split:]

print(f"Trainset: {len(trainset)} examples")
print(f"Devset: {len(devset)} examples")
print("\nSample train example:")
print(f"  Question: {trainset[0].question}")
print(f"  Answer: {trainset[0].answer}")

## 4. Writing a Custom Metric

In [ ]:
def answer_metric(example, prediction, trace=None):
    """Check if predicted answer contains expected answer."""
    expected = example.answer.lower()
    predicted = prediction.answer.lower()
    return 1.0 if expected in predicted or predicted in expected else 0.0


# Test the metric
class TestEx(dspy.Example):
    pass


ex = dspy.Example(question="What is RAG?", answer="Retrieval-Augmented Generation").with_inputs(
    "question"
)
pred = dspy.Prediction(answer="RAG stands for Retrieval-Augmented Generation")
print(f"Metric score: {answer_metric(ex, pred)}")

## 5. Baseline Program

In [ ]:
class QA(dspy.Signature):
    """Answer questions with short factual responses."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


class SimpleQA(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.ChainOfThought(QA)

    def forward(self, question):
        return self.generate(question=question)


baseline = SimpleQA()

# Evaluate baseline
evaluator = Evaluate(devset=devset, metric=answer_metric, num_threads=4, display_progress=True)
baseline_score = evaluator(baseline)
print(f"\nBaseline score: {baseline_score:.2f}")

## 6. BootstrapFewShot — Quick Baseline Optimizer

In [ ]:
# BootstrapFewShot: generates few-shot demos by running the program
teleprompter = BootstrapFewShot(metric=answer_metric, max_bootstrapped_demos=4)

optimized_bootstrap = teleprompter.compile(SimpleQA(), trainset=trainset)

# Evaluate optimized
bootstrap_score = evaluator(optimized_bootstrap)
print(f"\nBootstrapFewShot score: {bootstrap_score:.2f}")
print(f"Improvement: {bootstrap_score - baseline_score:+.2f}")

## 7. MIPROv2 — Bayesian Optimization

In [ ]:
# MIPROv2: Bayesian optimization of instructions + demos
# More expensive but generally better results

mipro = MIPROv2(
    metric=answer_metric,
    num_candidates=5,  # Reduced for cost
    init_temperature=1.0,
)

optimized_mipro = mipro.compile(
    SimpleQA(),
    trainset=trainset,
    num_trials=DSPY_OPTIMIZER_TRIALS,  # Configurable via .env
    valset=devset,
)

# Evaluate optimized
mipro_score = evaluator(optimized_mipro)
print(f"\nMIPROv2 score: {mipro_score:.2f}")
print(f"Improvement over baseline: {mipro_score - baseline_score:+.2f}")
print(f"Improvement over Bootstrap: {mipro_score - bootstrap_score:+.2f}")

## 8. Inspecting Generated Instructions

In [ ]:
# Inspect what the optimizer generated
print("Generated demos (BootstrapFewShot):")
print(f"  {len(optimized_bootstrap.generate.demos)} demos")
for i, demo in enumerate(optimized_bootstrap.generate.demos[:2]):
    print(f"\n  Demo {i + 1}:")
    print(f"    Question: {demo.question}")
    print(f"    Answer: {demo.answer}")

print("\n" + "=" * 50)
print("Generated instructions may be embedded in the compiled program.")
print("Use optimized_mipro.save() to inspect the full program.")

## 9. Saving and Reloading

In [ ]:
# Save optimized program
optimized_mipro.save("optimized_qa.json")
print("✅ Saved to optimized_qa.json")

# Reload
loaded = SimpleQA()
loaded.load("optimized_qa.json")

# Verify it works
result = loaded(question="What is DSPy?")
print(f"\nReloaded program answer: {result.answer}")

## 10. Exercise: Optimize a Classification Pipeline

Optimize a sentiment classifier on your own dataset using:
1. BootstrapFewShot as baseline
2. MIPROv2 for best results
3. Compare scores and inspect generated demos

In [ ]:
# YOUR TURN: Optimize a classification pipeline

# class Sentiment(dspy.Signature):
#     """Classify sentiment."""
#     text: str = dspy.InputField()
#     sentiment: str = dspy.OutputField()

# class SentimentClassifier(dspy.Module):
#     def __init__(self):
#         super().__init__()
#         self.classify = dspy.ChainOfThought(Sentiment)
#     def forward(self, text):
#         return self.classify(text=text)

# # Optimize
# teleprompter = MIPROv2(metric=your_metric)
# optimized = teleprompter.compile(SentimentClassifier(), trainset=trainset, num_trials=10)

---

**Next:** [03_rag_pipeline.ipynb](03_rag_pipeline.ipynb) — RAG with assertions and constraints